# 03 - DPO fine-tuning of Gemma 2 2B Instruct (Unsloth + QLoRA, Kaggle T4)

Same recipe as notebook 02, adapted to `google/gemma-2-2b-it`:
* Gemma is a **gated** model: add your Hugging Face token as a Kaggle secret named `HF_TOKEN`
  (and accept the licence on the model page).
* A shorter `MAX_SEQ_LENGTH` (850 vs 2000 tokens) is needed to fit the T4's 14.6 GB of VRAM;
  longer answers are therefore truncated (see the *Limitations* section of the thesis).

> Training data is not distributed with this repo - point `DATASET_PATH` at the train split
> produced by `01_create_dpo_dataset.ipynb` (Gemma was trained on the Best-vs-Worst dataset).

> 🔒 **Data availability.** The preference datasets used in this work (and the AutoBench export they are derived from) were shared privately by the AutoBench authors and are **not included in this repository**. All paths in this notebook are placeholders (`<your-...>`): point them at your own data with the same schema (`prompt`, `chosen`, `rejected`). Cell outputs have been cleared so that no private data is exposed.

## 1. Install Unsloth

In [ ]:
import subprocess, sys

# Unsloth installs the version matching the GPU that is present.
# Do NOT install bitsandbytes separately: Unsloth ships a patched build.
# Do NOT install transformers/trl separately: Unsloth pins compatible versions.
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "unsloth[kaggle-new]",   # Kaggle-optimised variant
    "--quiet", "--upgrade"
])

import unsloth
print(f"Unsloth {unsloth.__version__} installed")

## 2. Environment setup

In [ ]:
import os

# Force Kaggle to use only the first GPU (avoids odd behaviour with 2xT4)
os.environ["CUDA_VISIBLE_DEVICES"]    = "0"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_DIR  = "/kaggle/working"
CACHE_DIR = os.path.join(BASE_DIR, "hf_cache")

os.environ["HF_HOME"]            = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = os.path.join(CACHE_DIR, "transformers")
os.environ["HF_DATASETS_CACHE"]  = os.path.join(CACHE_DIR, "datasets")

os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Cache -> {CACHE_DIR}")

## 3. GPU check

In [ ]:
import torch
from unsloth import is_bfloat16_supported

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU            : {props.name}")
    print(f"VRAM           : {props.total_memory / 1024**3:.1f} GB")

# The T4 does NOT support BF16 - this flag is used in the training config
USE_BF16 = is_bfloat16_supported()
USE_FP16 = not USE_BF16
print(f"BF16 supported : {USE_BF16}")
print(f"FP16 used      : {USE_FP16}")

## 4. Configuration

In [ ]:
# NOTE: private data - not distributed with this repo. Replace the placeholder path below.
DATASET_PATH = "/kaggle/input/<your-dpo-dataset>/dpo_best_worst_train.parquet"

COL_PROMPT   = "prompt"
COL_CHOSEN   = "chosen"
COL_REJECTED = "rejected"

TRAINING_MODE = "dpo"
MODEL_ID      = "google/gemma-2-2b-it"

# -- Hyper-parameters ------------------------------------------
MAX_SEQ_LENGTH    = 850                   # max tokenised length (memory-bound on a T4)
MAX_PROMPT_LENGTH = MAX_SEQ_LENGTH // 2

NUM_EPOCHS        = 1
BATCH_SIZE        = 2
GRAD_ACCUM        = 8      # effective batch = 2 * 8 = 16 samples
LEARNING_RATE     = 5e-5
BETA              = 0.1
WARMUP_STEPS      = 0.1

# -- LoRA ------------------------------------------------------
LORA_R       = 16
LORA_ALPHA   = 16
LORA_DROPOUT = 0

# -- Output ----------------------------------------------------
OUTPUT_DIR = os.path.join(BASE_DIR, f"model-{TRAINING_MODE}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Training mode   : {TRAINING_MODE.upper()}")
print(f"Model           : {MODEL_ID}")
print(f"Output          : {OUTPUT_DIR}")
print(f"Max seq length  : {MAX_SEQ_LENGTH}")
print(f"Effective batch : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")

## 5. Load dataset

In [ ]:
from datasets import load_dataset

ext = DATASET_PATH.rsplit(".", 1)[-1].lower()
fmt = {"json": "json", "jsonl": "json", "csv": "csv", "parquet": "parquet"}.get(ext, "json")

raw = load_dataset(fmt, data_files=DATASET_PATH, split="train")

# Rename columns if necessary
rename = {}
for target, src in [("prompt", COL_PROMPT), ("chosen", COL_CHOSEN), ("rejected", COL_REJECTED)]:
    if src != target and src in raw.column_names:
        rename[src] = target
if rename:
    raw = raw.rename_columns(rename)

# Keep only the DPO columns
raw = raw.select_columns(["prompt", "chosen", "rejected"])

# 90/10 train/eval split - essential with a small dataset
split    = raw.train_test_split(test_size=0.1, seed=42)
ds_train = split["train"]
ds_eval  = split["test"]

print(f"Train: {len(ds_train)} examples")
print(f"Eval : {len(ds_eval)} examples")

## 6. Hugging Face authentication (gated model)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret = UserSecretsClient()
hf_token = secret.get_secret("HF_TOKEN")
login(token=hf_token)
print("Authenticated on Hugging Face")

## 7. Load the model with Unsloth (4-bit / QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,         # None = autodetect: FP16 on T4, BF16 on A100/H100
    load_in_4bit   = True,         # QLoRA 4-bit
    cache_dir      = CACHE_DIR,
    token          = hf_token,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"  # required for DPO

print(f"Model loaded: {MODEL_ID}")
print(f"  Pad token   : {tokenizer.pad_token!r}")
print(f"  Padding side: {tokenizer.padding_side}")

## 8. LoRA configuration

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",  # ~30% less VRAM than gradient_checkpointing=True
    random_state               = 42,
)

model.print_trainable_parameters()

## 9. Training (DPO)

In [ ]:
import torch
from unsloth import PatchDPOTrainer

# Mandatory patch: applies Unsloth's optimised kernels to DPOTrainer
PatchDPOTrainer()

torch.cuda.empty_cache()

SHARED_TRAINING_KWARGS = dict(
    output_dir                    = OUTPUT_DIR,
    num_train_epochs              = NUM_EPOCHS,
    per_device_train_batch_size   = BATCH_SIZE,
    per_device_eval_batch_size    = BATCH_SIZE,
    gradient_accumulation_steps   = GRAD_ACCUM,
    learning_rate                 = LEARNING_RATE,
    lr_scheduler_type             = "cosine",
    warmup_steps                  = WARMUP_STEPS,
    fp16                          = USE_FP16,       # True on T4, False on A100/H100
    bf16                          = USE_BF16,       # False on T4, True on A100/H100
    optim                         = "adamw_8bit",   # recommended by Unsloth
    eval_strategy                 = "steps",
    eval_steps                    = 16,
    save_strategy                 = "steps",
    save_steps                    = 16,
    save_total_limit              = 2,
    load_best_model_at_end        = False,
    logging_steps                 = 10,
    report_to                     = "none",
    remove_unused_columns         = False,
    dataloader_num_workers        = 0,
    dataloader_pin_memory         = False,
    max_length                    = MAX_SEQ_LENGTH,
    max_prompt_length             = MAX_PROMPT_LENGTH,
    beta                          = BETA,
    dataset_num_proc              = 4,              # parallel tokenisation on CPU
)

from trl import DPOTrainer, DPOConfig

config = DPOConfig(
    **SHARED_TRAINING_KWARGS,
    truncation_mode          = "keep_start",
    loss_type                = "sigmoid",
    label_smoothing          = 0.0,
    disable_dropout          = True,
    precompute_ref_log_probs = False,
)

trainer = DPOTrainer(
    model            = model,
    ref_model        = None,   # with LoRA the frozen base acts as the reference model
    args             = config,
    train_dataset    = ds_train,
    eval_dataset     = ds_eval,
    processing_class = tokenizer,
)

print(f"Starting {TRAINING_MODE.upper()} training with Unsloth...")
print(f"  Effective batch : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Epochs          : {NUM_EPOCHS}")
print(f"  FP16/BF16       : fp16={USE_FP16}, bf16={USE_BF16}")

train_result = trainer.train()

print("\nTraining results:")
print(train_result)

## 10. Save the LoRA adapters

In [ ]:
FINAL_DIR = os.path.join(OUTPUT_DIR, "final")
os.makedirs(FINAL_DIR, exist_ok=True)

# Save only the LoRA adapters (a few MB), not the base weights
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

# Save metrics and trainer state
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

print(f"LoRA adapter saved to: {FINAL_DIR}")
print("\nFiles:")
for f in sorted(os.listdir(FINAL_DIR)):
    size = os.path.getsize(os.path.join(FINAL_DIR, f)) / 1024**2
    print(f"  {f:45s} {size:.1f} MB")